# Tahap 2: Training & Evaluasi Collaborative Filtering

Pipeline utama untuk Evaluasi & Training Implicit Collaborative Filtering:
1. Evaluasi grid search menggunakan ALS (Alternating Least Squares)
2. Evaluasi grid search menggunakan BPR (Bayesian Personalized Ranking)
3. Fit kembali menggunakan metrik tebaik pada keseluruhan data `train + validation`
4. Uji dan tampilkan *test metric* pada `Leave-One-Out Test Setup`.

In [ ]:
import json
import pickle
import time
from pathlib import Path
from itertools import product
import pandas as pd

# Load custom components
from data_prep import (
    load_cf_splits,
    get_matrix_dims,
    build_user_item_matrix,
    build_user_history,
    prepare_loo_eval,
)
from evaluator import evaluate_loo, metrics_to_dataframe
from models.als_model import ALSModel
from models.bpr_model import BPRModel
from models.svd_model import SVDModel
from models.ncf_model import NCFModel

ROOT = Path("../../food.com")
OUTPUT_DIR = Path("outputs")
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"

for d in [MODEL_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

N_NEGATIVES = 99
K_VALUES = (5, 10, 20)
SEED = 42

## 1. Load Data Pembagian Matrix
Memuat matrix yang sudah digenerate dari `01_build_cf_split.py`. 

In [ ]:
print("Memuat Dataset train, test, validation...")
train_df, val_df, test_df = load_cf_splits()
n_users, n_items = get_matrix_dims(train_df, val_df, test_df)

print("Membangun matriks dan riwayat interaksi...")
train_matrix = build_user_item_matrix(train_df, n_users, n_items, binary=True)
print(f"Matrix User-Item: {train_matrix.shape}")

train_history = build_user_history(train_df)
train_val_history = build_user_history(train_df, val_df)

# Ekstrak Evaluasi LOO Negatives
val_loo = prepare_loo_eval(val_df, train_history, n_items, N_NEGATIVES, SEED)
test_loo = prepare_loo_eval(test_df, train_val_history, n_items, N_NEGATIVES, SEED)

print("Kesiapan matrix telah diselesaikan.")

## 2. Menjalankan Parameter Grid Search (Validation Phase)
Pada contoh tutorial `jupytext` ini, kita menggunakan setup hiperparameter ringkas supaya cepat berjalan. 

In [ ]:
def run_grid_search(ModelClass, param_grid, train_matrix, val_loo, model_name):
    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))
    
    print(f"\n[{model_name}] Memulai Grid Search...")
    best_hr10, best_params, best_model = -1.0, None, None
    all_results = []
    
    for combo in combos:
        params = dict(zip(keys, combo))
        print(f"Menguji: {params}")
        
        # Training
        model = ModelClass(**params)
        model.fit(train_matrix)
        
        # Validation
        metrics = evaluate_loo(model, val_loo, k_values=K_VALUES, verbose=False)
        all_results.append({**params, **metrics})
        
        if metrics["HR@10"] > best_hr10:
            best_hr10 = metrics["HR@10"]
            best_params = params
            best_model = model
            
    print(f"--> [Terbaik {model_name}] Validation HR@10: {best_hr10:.4f} @ {best_params}\n")
    return best_model, best_params, best_hr10, pd.DataFrame(all_results)

# Grid hyperparameters ringkas
ALS_GRID_QUICK = {
    "factors": [32, 64],
    "regularization": [0.01],
    "iterations": [20],
    "alpha": [20.0, 40.0],
}
BPR_GRID_QUICK = {
    "factors": [32, 64],
    "learning_rate": [0.01, 0.05],
    "regularization": [0.01],
    "iterations": [100],
}
SVD_GRID_QUICK = {
    "factors": [50],
    "lr_all": [0.005],
    "reg_all": [0.02],
    "epochs": [20],
    "num_neg_samples": [2],
}
NCF_GRID_QUICK = {
    "gmf_factors": [8],
    "mlp_factors": [8],
    "batch_size": [1024],
    "lr": [0.001],
    "epochs": [5],  # 5 epoch for quick smoke test
}

_, best_als_params, als_val_hr10, als_res = run_grid_search(ALSModel, ALS_GRID_QUICK, train_matrix, val_loo, "ALS")
_, best_bpr_params, bpr_val_hr10, bpr_res = run_grid_search(BPRModel, BPR_GRID_QUICK, train_matrix, val_loo, "BPR")
_, best_svd_params, svd_val_hr10, svd_res = run_grid_search(SVDModel, SVD_GRID_QUICK, train_matrix, val_loo, "SVD")
_, best_ncf_params, ncf_val_hr10, ncf_res = run_grid_search(NCFModel, NCF_GRID_QUICK, train_matrix, val_loo, "NCF")

## 3. Final Retraining & Testing
Mengajarkan `best metrics parameters` kembali pada gabungan data `train+val`, kemudian diuji di Test set.

In [ ]:
print("Melakukan Re-training model pada Train+Validation...")
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
train_val_matrix = build_user_item_matrix(train_val_df, n_users, n_items, binary=True)

final_als = ALSModel(**best_als_params)
final_als.fit(train_val_matrix)

final_bpr = BPRModel(**best_bpr_params)
final_bpr.fit(train_val_matrix)

final_svd = SVDModel(**best_svd_params)
final_svd.fit(train_val_matrix)

final_ncf = NCFModel(**best_ncf_params)
final_ncf.fit(train_val_matrix)

# Mendapatkan hasil evaluasi:
print("\n[EVALUASI TEST SET - ALS]")
als_test = evaluate_loo(final_als, test_loo, k_values=K_VALUES, verbose=True)

print("\n[EVALUASI TEST SET - BPR]")
bpr_test = evaluate_loo(final_bpr, test_loo, k_values=K_VALUES, verbose=True)

print("\n[EVALUASI TEST SET - SVD]")
svd_test = evaluate_loo(final_svd, test_loo, k_values=K_VALUES, verbose=True)

print("\n[EVALUASI TEST SET - NCF]")
ncf_test = evaluate_loo(final_ncf, test_loo, k_values=K_VALUES, verbose=True)

## 4. Menyimpan Model ke Log Repositori

In [ ]:
all_test = {"ALS": als_test, "BPR": bpr_test, "SVD": svd_test, "NCF": ncf_test}
results_df = metrics_to_dataframe(all_test)

print("\nKESIMPULAN PERBANDINGAN MODEL:")
print(results_df.to_string(float_format=lambda x: f"{x:.4f}"))

results_df.to_csv(RESULT_DIR / "final_results.csv")

# Tentukan model the best of the best
best_name = "BPR"
best_score = bpr_test["HR@10"]
best_final = final_bpr

if als_test["HR@10"] > best_score:
    best_name, best_score, best_final = "ALS", als_test["HR@10"], final_als
if svd_test["HR@10"] > best_score:
    best_name, best_score, best_final = "SVD", svd_test["HR@10"], final_svd
if ncf_test["HR@10"] > best_score:
    best_name, best_score, best_final = "NCF", ncf_test["HR@10"], final_ncf

with open(MODEL_DIR / f"best_cf_model_{best_name.lower()}.pkl", "wb") as f:
    pickle.dump({"name": best_name, "model": best_final}, f)

print(f"\nExport Artifact ({best_name}) Berhasil -> output folder.")